# Point centric validation

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# Point-Centric Pipeline Validation

This notebook validates the coastal-transformer preprocessing outputs:
- `X_dynamic`: temporally aligned offshore features
- `Y_targets`: per-NORAC target arrays
- `X_static`: per-NORAC static relation vectors

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists() and (REPO_ROOT.parent / "configs").exists():
    REPO_ROOT = REPO_ROOT.parent

OUT_DIR = REPO_ROOT / "data/processed/outerfjord_1"
X_PATH = OUT_DIR / "point_centric_X_dynamic.npz"
Y_PATH = OUT_DIR / "point_centric_Y_targets.npz"
S_PATH = OUT_DIR / "point_centric_X_static.npz"
M_PATH = OUT_DIR / "point_centric_metadata.json"

assert X_PATH.exists(), f"Missing: {X_PATH}"
assert Y_PATH.exists(), f"Missing: {Y_PATH}"
assert S_PATH.exists(), f"Missing: {S_PATH}"
assert M_PATH.exists(), f"Missing: {M_PATH}"

print("Kernel CWD:", Path.cwd())
print("Using output directory:", OUT_DIR)

In [ ]:
x_npz = np.load(X_PATH, allow_pickle=True)
y_npz = np.load(Y_PATH, allow_pickle=True)
s_npz = np.load(S_PATH, allow_pickle=True)
metadata = json.loads(M_PATH.read_text())

X_dynamic = x_npz["X_dynamic"]
timestamps = x_npz["timestamps"].astype(str)
dynamic_feature_names = x_npz["feature_names"].astype(str).tolist()

target_sites = y_npz["target_sites"].astype(str).tolist()
target_feature_names = y_npz["target_feature_names"].astype(str).tolist()
Y_targets = {site: y_npz[f"Y__{site}"] for site in target_sites}

static_feature_names = s_npz["static_feature_names"].astype(str).tolist()
X_static = {site: s_npz[f"Xstatic__{site}"] for site in target_sites}

train_idx = x_npz["train_idx"]
val_idx = x_npz["val_idx"]
test_idx = x_npz["test_idx"]

In [ ]:
# Required shape checks
print("X_dynamic shape:", X_dynamic.shape)
print("Number of dynamic feature names:", len(dynamic_feature_names))
print("Number of target sites:", len(target_sites))
print("Target feature count per site:", len(target_feature_names))
print("Static feature count per site:", len(static_feature_names))
print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))

print("\nY_targets shapes:")
for site in target_sites:
    print(f"  {site}: {Y_targets[site].shape}")

print("\nX_static shapes:")
for site in target_sites:
    print(f"  {site}: {X_static[site].shape}")

# Sanity: all Y arrays have same temporal length as X_dynamic
for site in target_sites:
    assert Y_targets[site].shape[0] == X_dynamic.shape[0], f"Time mismatch for {site}"

print("\nShape validation passed.")

In [ ]:
# Temporal alignment plot: one offshore hs-like channel vs one nearshore hs channel
plot_site = target_sites[0]
plot_steps = min(400, X_dynamic.shape[0])

dyn_idx = next((i for i, n in enumerate(dynamic_feature_names) if n.endswith("_hs")), None)
if dyn_idx is None:
    dyn_idx = 0

try:
    tgt_idx = target_feature_names.index("hs")
except ValueError:
    tgt_idx = 0

ts = np.array(timestamps[:plot_steps], dtype="datetime64[ns]")
x_series = X_dynamic[:plot_steps, dyn_idx]
y_series = Y_targets[plot_site][:plot_steps, tgt_idx]

plt.figure(figsize=(13, 4))
plt.plot(ts, x_series, label=f"X_dynamic[{dynamic_feature_names[dyn_idx]}]", linewidth=1.2)
plt.plot(
    ts, y_series, label=f"Y_targets[{plot_site}][{target_feature_names[tgt_idx]}]", linewidth=1.2
)
plt.title("Temporal Alignment Check: Offshore vs Nearshore Slice")
plt.xlabel("Time")
plt.ylabel("Normalized Value")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Circular feature bounds sanity check (expected in [-1, 1])
def circular_bounds(arr, names, label):
    idx = [i for i, n in enumerate(names) if n.endswith("_sin") or n.endswith("_cos")]
    if not idx:
        print(f"{label}: no circular columns found")
        return
    vals = arr[:, idx] if arr.ndim == 2 else arr[idx]
    mn = float(np.nanmin(vals))
    mx = float(np.nanmax(vals))
    print(f"{label}: min={mn:.6f}, max={mx:.6f}, n_cols={len(idx)}")


circular_bounds(X_dynamic, dynamic_feature_names, "X_dynamic circular")

Y_stack = np.vstack([Y_targets[s] for s in target_sites])
circular_bounds(Y_stack, target_feature_names, "Y_targets circular (stacked)")

S_stack = np.vstack([X_static[s] for s in target_sites])
circular_bounds(S_stack, static_feature_names, "X_static circular (stacked)")

print("\nMetadata circular groups (counts + sample names):")
for group_name, cols in (metadata.get("circular_features", {}) or {}).items():
    cols = list(cols)
    print(f"  {group_name}: count={len(cols)}, sample={cols[:5]}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

# Resolve repo root when kernel may start from /notebooks
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists() and (REPO_ROOT.parent / "configs").exists():
    REPO_ROOT = REPO_ROOT.parent

S_PATH = REPO_ROOT / "data/processed/coastal_dataset_1" / "point_centric_X_static.npz"
print("Static NPZ path:", S_PATH)
npz = np.load(S_PATH, allow_pickle=True)
static_feature_names = npz["static_feature_names"].astype(str).tolist()
sites = npz["target_sites"].astype(str).tolist()
print("feature count (all):", len(static_feature_names))
print("num_sites:", len(sites))

# Filter out any features that refer to wind grids so we only preview NORA3 grid-derived features
mask_no_wind = [("wind" not in n.lower()) for n in static_feature_names]
num_removed = len(static_feature_names) - sum(mask_no_wind)
print(f"Removed {num_removed} wind-related static features (if any).")
filtered_feature_names = [n for n, keep in zip(static_feature_names, mask_no_wind) if keep]

if not filtered_feature_names:
    print("No non-wind static features found to preview.")
else:
    # Show a compact labeled preview for the first 3 sites, using only non-wind features
    for site in sites[:3]:
        arr = npz[f"Xstatic__{site}"].astype(float)
        arr_filtered = arr[np.array(mask_no_wind, dtype=bool)]
        print("\nSite:", site, "shape_raw:", arr.shape, "shape_filtered:", arr_filtered.shape)
        preview_n = min(16, arr_filtered.size)
        for i in range(preview_n):
            print(f"  {filtered_feature_names[i]:30s} : {arr_filtered[i]:.6f}")
        if arr_filtered.size > preview_n:
            print("  ...")
            for i in range(arr_filtered.size - 4, arr_filtered.size):
                print(f"  {filtered_feature_names[i]:30s} : {arr_filtered[i]:.6f}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Resolve repo root when kernel may start from /notebooks
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists() and (REPO_ROOT.parent / "configs").exists():
    REPO_ROOT = REPO_ROOT.parent

S_PATH = REPO_ROOT / "data/processed/coastal_dataset_1" / "point_centric_X_static.npz"
print("Static NPZ path:", S_PATH)
npz = np.load(S_PATH, allow_pickle=True)
static_feature_names = npz["static_feature_names"].astype(str).tolist()
sites = npz["target_sites"].astype(str).tolist()

try:
    depth_idx = static_feature_names.index("norac_depth_m")
except ValueError:
    depth_idx = None

if depth_idx is None:
    print("norac_depth_m not found in static_feature_names")
else:
    rows = []
    for site in sites:
        arr = npz[f"Xstatic__{site}"].astype(float)
        depth = arr[depth_idx]
        rows.append({"site": site, "norac_depth_m": depth})
    df = pd.DataFrame(rows)
    # Show up to 20 sites in a compact table
    print(df.to_string(index=False, max_rows=20))